In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas

In [ ]:
import gc
import torch
import pandas as pd
import torch
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

# Check GPU availability
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
# importing models
base_repo = "NousResearch/Llama-2-7b-chat-hf"
ft_repo   = "sk122003/biomat-chat-finetune"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

def load_model(repo):
    tokenizer = AutoTokenizer.from_pretrained(repo)
    model = AutoModelForCausalLM.from_pretrained(
        repo, quantization_config=bnb_config, device_map={"": 0}
    )
    model.config.use_cache = True
    model.eval()
    return tokenizer, model

base_tok, base_model = load_model(base_repo)
ft_tok, ft_model     = load_model(ft_repo)

print("Both models loaded.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Both models loaded.


In [ ]:

questions = [
    "What is a biomaterial?",
    "I’m designing a scaffold for load-bearing bone repair, recommend a suitable material with reasons.",
    "How do the mechanical properties and degradation rates of synthetic and natural polymers influence their suitability for cartilage tissue engineering?",
    # ...
]
#answer generations
def generate_answers(tokenizer, model, questions, max_new_tokens=256):
    answers = []
    for q in questions:
        prompt = f"[INST] {q} [/INST]"
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
            )

        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        answers.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
    return answers

base_answers = generate_answers(base_tok, base_model, questions)
ft_answers   = generate_answers(ft_tok, ft_model, questions)

df = pd.DataFrame({
    "question": questions,
    "base": base_answers,
    "fine_tuned": ft_answers,
})

pd.set_option("display.max_colwidth", None)
df.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

,question,base,fine_tuned
0,What is a biomaterial?,"A biomaterial is a material that is used in contact with or inside the human body to provide a medical benefit. everybody has heard of plastic, metal, and ceramic, but biomaterials are different. they are designed to interact with the body's tissues and biological processes in a way that is safe and effective. biomaterials can be used to repair damaged tissues, replace missing tissues, or even restore function to damaged tissues. Biomaterials can be made from a variety of materials, including: 1. Biological materials: These are materials that are derived from living organisms, such as collagen, fibrin, and hyaluronic acid. 2. Synthetic materials: These are materials that are created in a laboratory, such as polyurethane, polyester, and silicone. 3. Natural materials: These are materials that are derived from natural sources, such as wood, bone, and cornstarch. 4. Hybrid materials: These are materials that combine biological and synthetic materials, such as a collagen-based scaffold with a silicone coating. Biom","A biomaterial is a material that is used in medical applications, such as implants, prosthetics, and drug delivery systems. The term biomaterial refers to any material that interacts with biological systems, including cells, tissues, and organs. Biomaterials can be natural or synthetic, and their properties are determined by their chemical composition, structure, and surface characteristics. Biomaterials are used in various applications, including tissue engineering, drug delivery, and biosensors."
1,"I’m designing a scaffold for load-bearing bone repair, recommend a suitable material with reasons.","When designing a scaffold for load-bearing bone repair, it's essential to choose a material that can withstand the mechanical forces and stresses applied to the repair site. Here are some suitable materials for load-bearing bone repair scaffolds, along with their reasons: 1. Titanium mesh: Titanium is a strong, lightweight, and corrosion-resistant metal that is well-suited for load-bearing bone repair. Titanium mesh is a popular choice for scaffolds because it provides excellent mechanical stability and can be easily shaped to fit the contours of the bone. Additionally, titanium is non-toxic and biocompatible, reducing the risk of adverse tissue reactions. 2. Polymethylmethacrylate (PMMA): PMMA is a rigid, biocompatible material that is commonly used in bone repair applications. It has high mechanical strength and can be molded into various shapes to fit the bone defect. PMMA is also non-toxic and biodegradable, making it a suitable choice for load-bearing bone repair.","A suitable material for load-bearing bone repair is a biodegradable polymer, such as polylactic acid (PLA), polycaprolactone (PCL), or polyurethane (PU). These materials are biocompatible, biodegradable, and can be tailored to release growth factors to promote bone regeneration. PLA is a popular choice due to its excellent mechanical properties, biocompatibility, and biodegradability. It has a high tensile strength, elongation at break, and modulus, making it suitable for load-bearing applications."
2,How do the mechanical properties and degradation rates of synthetic and natural polymers influence their suitability for cartilage tissue engineering?,"Synthetic and natural polymers have different mechanical properties and degradation rates, which can influence their suitability for cartilage tissue engineering. everybody. Mechanical Properties: 1. Synthetic polymers: * Have higher mechanical strength and stiffness than natural polymers. * Can withstand higher loads and stresses without deforming or breaking. * Are more resistant to wear and tear. * Can be tailored to have specific mechanical properties, such as elasticity or toughness. 2. Natural polymers: * Have lower mechanical strength and stiffness than synthetic polymers. * Are more flexible and can deform under load, which can be beneficial for tiss